# Workshop Agent 7: The Healthcare Intelligence Agent

**Workshop:** *10 Essential AI Agents Every Engineer Must Build*  
**Companion to:** *30 Agents Every AI Engineer Must Build* — Imran Ahmad (Packt Publishing, 2026)  
**Book Reference:** Chapter 13, §13.1–13.4 (pp. 362–375)

---

> *"The very first requirement in a hospital is that it should do the sick no harm."*
> — Florence Nightingale, *Notes on Hospitals* (1863)

In this session you will build a multi-agent diagnostic-assistance system modeled on a real regional health-network deployment. Patient inputs — vitals, reported symptoms, and FHIR-normalized history — flow through a four-layer architecture (data ingestion, clinical knowledge, reasoning and decision, explanation and delivery) and produce a clinical decision-support output with its clinical reasoning fully surfaced. You will implement the Bayesian belief update that advances the agent's POMDP belief state (§13.1), a clinical knowledge base with provenance tracking (§13.1), a FHIR normalization and patient data pipeline (§13.2), and a diagnostic coordinator that calibrates confidence, escalates critical conditions above a 0.15 safety threshold, and generates audience-adapted explanations for clinician and patient alike (§13.3). The session closes with the deployment case study from §13.4: a 200,000-patient regional network reporting a 30% improvement in early detection and 92% physician satisfaction.

**Simulation Mode:** This notebook runs fully without API keys. All external dependencies have deterministic mock fallbacks derived from Chapter 13 content.


In [ ]:
# Google Colab bootstrap — runs only on Colab, no-op everywhere else.
# Locally you are already inside the agent folder with requirements installed.
import os
import sys

if "google.colab" in sys.modules:
    AGENT_DIR = "07-healthcare-intelligence-agent"
    if not os.path.exists("/content/repo"):
        os.system("git clone --depth 1 https://github.com/cloudanum/ws-10-agents /content/repo")
    os.chdir(f"/content/repo/{AGENT_DIR}")
    # On Colab, prefer requirements-colab.txt when present: it drops pins that
    # cannot coexist with Colab's preinstalled stack (e.g. langchain 0.2.16
    # requires numpy<2 on Python 3.13, while Colab ships numpy 2.x).
    req_file = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
    # Keep Colab's preinstalled scientific/kernel stack: the kernel already has
    # numpy, pandas, pydantic and ipykernel loaded, so letting pip replace them
    # (e.g. building numpy 1.26.4 from source or upgrading ipykernel) breaks the
    # running kernel with ABI errors or an OOM kill. Filter those lines out of
    # requirements and constrain the rest of the install to the installed versions.
    import re
    from importlib.metadata import PackageNotFoundError, version
    filtered = [
        line for line in open(req_file)
        if not re.match(r"\s*(numpy|pandas|pydantic|jupyter|ipykernel)\b", line, re.IGNORECASE)
    ]
    with open("/tmp/colab_requirements.txt", "w") as fh:
        fh.writelines(filtered)
    pins = []
    for pkg in ("numpy", "pandas", "pydantic", "ipykernel"):
        try:
            pins.append(f"{pkg}=={version(pkg)}")
        except PackageNotFoundError:
            pass
    with open("/tmp/colab_constraints.txt", "w") as fh:
        fh.write("\n".join(pins) + "\n")
    import subprocess
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", "/tmp/colab_requirements.txt",
         "--constraint", "/tmp/colab_constraints.txt"],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print("pip install failed — re-running without -q for the full resolver report:\n")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "-r", "/tmp/colab_requirements.txt",
             "--constraint", "/tmp/colab_constraints.txt"],
        )
        raise RuntimeError("Colab bootstrap: pip install failed (see resolver report above)")
    print(f"Colab setup complete ({req_file}) — working directory: {os.getcwd()}")
else:
    print("Not on Colab — skipping bootstrap (local setup already in place).")

## Section 0: Setup & Configuration
*Ref: Technical Requirements, p. 362*

In [1]:
# Cell 0.2 — Imports
# Ref: Technical Requirements, p. 362
# Author: Imran Ahmad

import os
import sys
import json
import asyncio
import functools
import warnings
import hashlib
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Callable, Tuple

import numpy as np

# Conditional imports for optional dependencies
try:
    from dotenv import load_dotenv
    _HAS_DOTENV = True
except ImportError:
    _HAS_DOTENV = False

try:
    import nest_asyncio
    nest_asyncio.apply()
    _HAS_NEST_ASYNCIO = True
except ImportError:
    _HAS_NEST_ASYNCIO = False

# Suppress noisy warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("All imports loaded successfully.")
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")


All imports loaded successfully.
Python 3.14.5 | NumPy 2.5.2


In [2]:
# Multi-provider LLM support (OpenAI / Anthropic / Google Gemini)
# Set LLM_PROVIDER in .env to choose: openai | anthropic | google | auto
# Auto-detection uses the first available key.
# See supporting/llm_provider.py for details.

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

try:
    from supporting.llm_provider import detect_provider, get_llm, PROVIDER_MODELS, print_provider_banner
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = detect_provider()
    print_provider_banner(_PROVIDER, _PROVIDER_MODE)
except ImportError:
    print('[INFO] supporting/llm_provider.py not found — using default OpenAI path')
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = 'openai', os.getenv('OPENAI_API_KEY'), 'LIVE' if os.getenv('OPENAI_API_KEY') else 'SIMULATION'



   SIMULATION MODE ACTIVE
   Using MockLLM — no API key required



In [3]:
# Cell 0.4 — Color-Coded Logging Infrastructure
# Ref: Cross-cutting resilience pattern
# Author: Imran Ahmad

class ColorLog:
    """ANSI color codes for notebook log output."""
    BLUE  = '\033[94m'
    GREEN = '\033[92m'
    RED   = '\033[91m'
    YELLOW = '\033[93m'
    BOLD  = '\033[1m'
    RESET = '\033[0m'

def log_info(message: str) -> None:
    """Blue informational message."""
    print(f"{ColorLog.BLUE}{ColorLog.BOLD}[INFO]{ColorLog.RESET} "
          f"{ColorLog.BLUE}{message}{ColorLog.RESET}")

def log_success(message: str) -> None:
    """Green success message."""
    print(f"{ColorLog.GREEN}{ColorLog.BOLD}[SUCCESS]{ColorLog.RESET} "
          f"{ColorLog.GREEN}{message}{ColorLog.RESET}")

def log_error(message: str) -> None:
    """Red handled-error message."""
    print(f"{ColorLog.RED}{ColorLog.BOLD}[HANDLED ERROR]{ColorLog.RESET} "
          f"{ColorLog.RED}{message}{ColorLog.RESET}")

def log_warning(message: str) -> None:
    """Yellow warning message."""
    print(f"{ColorLog.YELLOW}{ColorLog.BOLD}[WARNING]{ColorLog.RESET} "
          f"{ColorLog.YELLOW}{message}{ColorLog.RESET}")

# Quick test
log_info("Color-coded logging initialized.")
log_success("Logging test passed.")


[INFO] Color-coded logging initialized.
[SUCCESS] Logging test passed.


In [4]:
# Cell 0.3 — API Key Management (Zero-Hardcode Policy)
# Ref: Cross-cutting — .env → getpass → Simulation Mode
# Author: Imran Ahmad

if _HAS_DOTENV:
    load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    try:
        if sys.stdin.isatty():
            import getpass
            OPENAI_API_KEY = getpass.getpass(
                "Enter your OpenAI API key (or press Enter for Simulation Mode): "
            )
    except Exception:
        pass  # Non-interactive environment — proceed to Simulation Mode

SIMULATION_MODE = not bool(OPENAI_API_KEY)

if SIMULATION_MODE:
    log_info(
        "SIMULATION MODE ACTIVE — All LLM calls use context-aware "
        "mock responses derived from Chapter 13 content."
    )
else:
    log_success(
        f"Live API mode active. Key loaded (ends ...{OPENAI_API_KEY[-4:]})."
    )


[INFO] SIMULATION MODE ACTIVE — All LLM calls use context-aware mock responses derived from Chapter 13 content.


## Section 1: Simulation Infrastructure
*Ref: Cross-cutting — enables full notebook execution without API keys*

This section builds the deterministic mock layer that powers Simulation Mode. Every external dependency — LLM calls, database queries, FHIR validators, literature APIs — has a chapter-derived fallback defined here.

In [5]:
# Cell 1.1 — MockResponse Dataclass
# Ref: Cross-cutting simulation infrastructure
# Author: Imran Ahmad

@dataclass
class MockResponse:
    """Standardized response container for all mock LLM calls."""
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)

log_info("MockResponse dataclass defined.")


[INFO] MockResponse dataclass defined.


In [6]:
# Cell 1.2 — MockLLM Class with Context-Aware Response Registry
# Ref: §13.1, §13.3, §13.5–13.7 — domain-specific mock responses
# Author: Imran Ahmad

class MockLLM:
    """
    Context-aware mock LLM returning domain-appropriate responses.

    Chapter 13: Healthcare and Scientific Agents
    Book: 30 Agents Every AI Engineer Must Build
    Author: Imran Ahmad
    """

    def __init__(self):
        self.call_count = 0
        self._response_registry = {
            "diagnostic": {
                "response": (
                    "Based on presented symptoms (fever 38.9C, tachycardia "
                    "118 bpm, elevated WBC 18.4), differential diagnosis "
                    "suggests: 1) Urosepsis (p=0.61), 2) Pneumonia-source "
                    "sepsis (p=0.21), 3) Biliary source (p=0.11)."
                ),
                "section": "13.3 Clinical Decision Support"
            },
            "drug_interaction": {
                "response": (
                    "WARNING: Concurrent use of warfarin and aspirin "
                    "increases bleeding risk. Recommend INR monitoring "
                    "every 48 hours per AHA guideline v2024.2."
                ),
                "section": "13.1 Medical Knowledge Integration"
            },
            "patient_summary": {
                "response": (
                    "Your temperature, heart rate, and blood test results "
                    "together suggest your body may be fighting a serious "
                    "infection. Your care team has been notified."
                ),
                "section": "13.3 Audience-Adapted Explanation"
            },
            "literature_synthesis": {
                "response": (
                    "Cluster analysis of 12,347 papers reveals 47 thematic "
                    "groups. Dominant clusters: aromatic polyimide synthesis "
                    "(n=2,841), nanocomposite reinforcement (n=1,923), "
                    "processing-property relationships (n=1,456)."
                ),
                "section": "13.5 Literature Synthesis"
            },
            "knowledge_gap": {
                "response": (
                    "Gap identified: block copolymer architectures with "
                    "thermally stable aromatic monomers. P(referenced)=0.73, "
                    "P(directly_studied)=0.04. Novelty: 0.89, Feasibility: 0.71."
                ),
                "section": "13.6 Knowledge Gap Identification"
            },
            "hypothesis": {
                "response": (
                    "Hypothesis H1: Alternating aromatic dianhydride-diamine "
                    "block copolymer with segment length 15-20 repeat units "
                    "will achieve Tg > 350C with elongation at break > 15%. "
                    "Testability score: 0.82."
                ),
                "section": "13.7 Hypothesis Generation"
            },
        }

    def invoke(self, prompt: str, context_type: str = "diagnostic") -> MockResponse:
        """Return a domain-appropriate mock response."""
        self.call_count += 1
        entry = self._response_registry.get(
            context_type, self._response_registry["diagnostic"]
        )
        log_info(
            f"[SIMULATION] MockLLM call #{self.call_count} | "
            f"context: {context_type} | source: {entry['section']}"
        )
        return MockResponse(
            content=entry["response"],
            metadata={"simulation": True, "section": entry["section"]}
        )

# Initialize the global LLM handle
if SIMULATION_MODE:
    llm = MockLLM()
    log_info("MockLLM initialized with 6-context response registry.")
else:
    llm = None  # Placeholder — live mode would instantiate ChatOpenAI here
    log_success("Live LLM client placeholder ready.")


[INFO] MockLLM initialized with 6-context response registry.


In [7]:
# Cell 1.3 — Mock Data Constants (9 Datasets)
# Ref: §13.1–13.8 — chapter-derived synthetic data for Simulation Mode
# Author: Imran Ahmad

# ─── Dataset 1: Patient Vitals (§13.1, §13.3) ───
# Source: Sepsis scenario from clinician report, p. 374
MOCK_PATIENT_VITALS = {
    "patient_id": "SIM-PT-00421",
    "temperature_c": 38.9,
    "heart_rate_bpm": 118,
    "blood_pressure_systolic": 92,
    "blood_pressure_diastolic": 58,
    "wbc_count": 18.4,
    "wbc_left_shift": True,
    "spo2_percent": 94,
    "map_mmhg": 65,
    "lactate_mmol": 2.8,
    "_source": "13.3 Sepsis scenario from clinician report"
}

# ─── Dataset 2: Diagnoses and Prior Belief (§13.1) ───
# Source: Bayesian belief update example, p. 363
MOCK_DIAGNOSES = [
    "urosepsis", "pneumonia_sepsis", "biliary_sepsis",
    "viral_syndrome", "dehydration"
]
MOCK_PRIOR_BELIEF = np.array([0.25, 0.25, 0.15, 0.20, 0.15])

# ─── Dataset 3: Likelihood Model (§13.1) ───
# Likelihood scores for each diagnosis given sepsis-like vitals
MOCK_LIKELIHOOD_SCORES = {
    "urosepsis":        0.82,
    "pneumonia_sepsis":  0.65,
    "biliary_sepsis":    0.45,
    "viral_syndrome":    0.20,
    "dehydration":       0.15
}

# ─── Dataset 4: Drug Interaction Records (§13.1) ───
# Source: Medical Knowledge Integration, pp. 365–367
MOCK_DRUG_DB = [
    {
        "drug_pair": ("warfarin", "aspirin"),
        "severity": "HIGH",
        "mechanism": "Additive anticoagulant effect",
        "recommendation": "Monitor INR every 48h",
        "source": "DrugBank v5.1.11",
        "guideline": "AHA 2024.2",
        "provenance": {
            "source": "drugbank",
            "version": "5.1.11",
            "retrieved_at": "2026-03-15T08:00:00Z",
            "confidence": 0.97
        }
    },
    {
        "drug_pair": ("metformin", "contrast_dye"),
        "severity": "MODERATE",
        "mechanism": "Risk of lactic acidosis",
        "recommendation": "Hold metformin 48h pre/post contrast",
        "source": "FDA Label 2026-revision",
        "guideline": "ACR Manual on Contrast Media 2026",
        "provenance": {
            "source": "fda_labels",
            "version": "2026.1",
            "retrieved_at": "2026-03-15T08:00:00Z",
            "confidence": 0.95
        }
    }
]

# ─── Dataset 5: FHIR Bundle (§13.2) ───
# Source: Patient data pipeline, pp. 367–369
MOCK_FHIR_BUNDLE = {
    "resourceType": "Bundle",
    "type": "collection",
    "entry": [
        {
            "resource": {
                "resourceType": "Patient",
                "id": "SIM-PT-00421",
                "name": [{"family": "Simulation", "given": ["Test"]}],
                "gender": "male",
                "birthDate": "1958-07-15"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "id": "obs-temp-001",
                "code": {"coding": [{"system": "http://loinc.org",
                         "code": "8310-5",
                         "display": "Body temperature"}]},
                "valueQuantity": {"value": 38.9, "unit": "Cel"},
                "effectiveDateTime": "2026-03-30T10:30:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Observation",
                "id": "obs-hr-001",
                "code": {"coding": [{"system": "http://loinc.org",
                         "code": "8867-4",
                         "display": "Heart rate"}]},
                "valueQuantity": {"value": 118, "unit": "/min"},
                "effectiveDateTime": "2026-03-30T10:30:00Z"
            }
        },
        {
            "resource": {
                "resourceType": "Condition",
                "id": "cond-dm2-001",
                "code": {"coding": [{"system": "http://snomed.info/sct",
                         "code": "44054006",
                         "display": "Type 2 diabetes mellitus"}]},
                "clinicalStatus": {"coding": [{"code": "active"}]},
                "onsetDateTime": "2018-04-01"
            }
        }
    ]
}

# ─── Dataset 6: Deployment Metrics (§13.4) ───
# Source: Diagnostic Assistance Case Study, pp. 374–375
MOCK_DEPLOYMENT_METRICS = {
    "patient_population": 200_000,
    "provider_sites": 20,
    "early_detection_improvement_pct": 30,
    "false_alarm_rate_pct": 3,
    "clinician_response_improvement_pct": 40,
    "physician_satisfaction_pct": 92,
    "differential_privacy_epsilon": 1.0,
    "raw_data_rate_kb_per_sec": 4,
    "derived_feature_size_bytes": 200,
    "transmission_interval_min": 15,
    "_source": "13.4 Diagnostic Assistance Case Study"
}

# ─── Dataset 7: Paper Corpus (§13.5) ───
# Source: Literature Synthesis, pp. 376–378
MOCK_PAPER_CORPUS = [
    {
        "doi": "10.1234/sim-polymer-001",
        "title": "Thermal Stability of Aromatic Polyimides: A Comprehensive Review",
        "authors": ["Zhang, L.", "Kumar, R."],
        "journal": "Progress in Polymer Science",
        "year": 2024,
        "abstract": "This review covers recent advances in aromatic polyimide thermal stability...",
        "citations": 187,
        "cluster": "aromatic_polyimide_synthesis",
        "source_db": "scopus"
    },
    {
        "doi": "10.1234/sim-polymer-002",
        "title": "Nanocomposite Reinforcement Strategies for High-Temperature Polymers",
        "authors": ["Chen, W.", "Patel, S.", "Yamamoto, K."],
        "journal": "Composites Science and Technology",
        "year": 2024,
        "abstract": "Nanocomposite reinforcement of polyimide matrices using...",
        "citations": 142,
        "cluster": "nanocomposite_reinforcement",
        "source_db": "ieee"
    },
    {
        "doi": "10.1234/sim-polymer-003",
        "title": "Processing-Property Relationships in Aerospace Polymer Systems",
        "authors": ["Williams, J.", "Dubois, M."],
        "journal": "Journal of Applied Polymer Science",
        "year": 2023,
        "abstract": "Systematic investigation of how processing conditions affect...",
        "citations": 98,
        "cluster": "processing_property",
        "source_db": "pubmed"
    },
    {
        "doi": "10.1234/sim-polymer-004",
        "title": "Block Copolymer Self-Assembly for Mechanical Property Optimization",
        "authors": ["Hernandez, A.", "Kim, T."],
        "journal": "Macromolecules",
        "year": 2024,
        "abstract": "Block copolymer architectures enable precise control of mechanical...",
        "citations": 211,
        "cluster": "block_copolymer_mechanical",
        "source_db": "scopus"
    },
    {
        "doi": "10.1234/sim-polymer-005",
        "title": "High-Temperature Homopolymer Architectures: State of the Art",
        "authors": ["Singh, P.", "Tanaka, H."],
        "journal": "Polymer Reviews",
        "year": 2023,
        "abstract": "Aromatic homopolymers with rigid backbone structures...",
        "citations": 156,
        "cluster": "high_temp_homopolymer",
        "source_db": "arxiv"
    },
    {
        "doi": "10.1234/sim-polymer-006",
        "title": "Polyimide Synthesis via Controlled Polycondensation",
        "authors": ["Li, X.", "Okonkwo, E."],
        "journal": "European Polymer Journal",
        "year": 2026,
        "abstract": "Novel controlled polycondensation routes for aromatic polyimide...",
        "citations": 63,
        "cluster": "aromatic_polyimide_synthesis",
        "source_db": "scopus"
    },
    {
        "doi": "10.1234/sim-polymer-007",
        "title": "Carbon Nanotube-Polyimide Nanocomposites for Aerospace Applications",
        "authors": ["Park, J.", "Garcia, R.", "Novak, L."],
        "journal": "ACS Applied Materials & Interfaces",
        "year": 2024,
        "abstract": "Incorporation of functionalized carbon nanotubes into polyimide...",
        "citations": 178,
        "cluster": "nanocomposite_reinforcement",
        "source_db": "pubmed"
    },
    {
        "doi": "10.1234/sim-polymer-008",
        "title": "Thermal-Mechanical Coupling in Aromatic Block Copolymers",
        "authors": ["Thompson, D.", "Wu, Y."],
        "journal": "Polymer",
        "year": 2023,
        "abstract": "Investigation of thermal-mechanical coupling effects in...",
        "citations": 89,
        "cluster": "block_copolymer_mechanical",
        "source_db": "ieee"
    },
    {
        "doi": "10.1234/sim-polymer-009",
        "title": "UV Degradation Kinetics of Polymer Coatings: Environmental Factors",
        "authors": ["Brown, K.", "Ahmed, F."],
        "journal": "Polymer Degradation and Stability",
        "year": 2024,
        "abstract": "Humidity effects on UV-induced polymer degradation remain...",
        "citations": 134,
        "cluster": "processing_property",
        "source_db": "scopus"
    },
    {
        "doi": "10.1234/sim-polymer-010",
        "title": "Rigid Rod Polyimides: Thermal Stability Beyond 400°C",
        "authors": ["Suzuki, M.", "Anderson, B."],
        "journal": "High Performance Polymers",
        "year": 2026,
        "abstract": "Rigid rod aromatic polyimides demonstrate exceptional thermal...",
        "citations": 47,
        "cluster": "high_temp_homopolymer",
        "source_db": "arxiv"
    },
    {
        "doi": "10.1234/sim-polymer-011",
        "title": "Creep Behavior of Nanocomposite-Reinforced Polyimides Under Cyclic Loading",
        "authors": ["Petrov, V.", "Nakamura, S."],
        "journal": "Mechanics of Materials",
        "year": 2022,
        "abstract": "Long-term creep behavior under cyclic thermal loading...",
        "citations": 201,
        "cluster": "nanocomposite_reinforcement",
        "source_db": "ieee"
    },
    {
        "doi": "10.1234/sim-polymer-012",
        "title": "Dianhydride Selection for Aerospace-Grade Polyimide Films",
        "authors": ["Costa, L.", "Wang, Z."],
        "journal": "Journal of Polymer Science",
        "year": 2024,
        "abstract": "Systematic screening of aromatic dianhydride monomers for...",
        "citations": 112,
        "cluster": "aromatic_polyimide_synthesis",
        "source_db": "pubmed"
    },
    {
        "doi": "10.1234/sim-polymer-013",
        "title": "Polystyrene-Polybutadiene Block Copolymers: Comprehensive Property Maps",
        "authors": ["Johnson, R.", "Bai, H."],
        "journal": "Macromolecular Chemistry and Physics",
        "year": 2023,
        "abstract": "Commodity block copolymer property maps as baseline...",
        "citations": 167,
        "cluster": "block_copolymer_mechanical",
        "source_db": "scopus"
    },
    {
        "doi": "10.1234/sim-polymer-014",
        "title": "Processing Windows for High-Tg Polymer Film Casting",
        "authors": ["Müller, T.", "Rao, V."],
        "journal": "Polymer Engineering & Science",
        "year": 2024,
        "abstract": "Identification of optimal processing windows for achieving...",
        "citations": 76,
        "cluster": "processing_property",
        "source_db": "ieee"
    },
    {
        "doi": "10.1234/sim-polymer-015",
        "title": "Aromatic Diamine Monomers for Next-Generation Aerospace Polymers",
        "authors": ["Ivanova, O.", "Lee, C."],
        "journal": "Chemistry of Materials",
        "year": 2026,
        "abstract": "Novel aromatic diamine monomers enabling higher thermal...",
        "citations": 55,
        "cluster": "high_temp_homopolymer",
        "source_db": "pubmed"
    }
]

# ─── Dataset 8: Gap Report (§13.6) ───
# Source: Knowledge Gap Identification, pp. 380–381
MOCK_GAP_REPORT = {
    "gaps": [
        {
            "id": "GAP-001",
            "description": (
                "Block copolymer architectures with thermally "
                "stable aromatic monomers for aerospace applications"
            ),
            "strategy": "cross_domain_intersection",
            "p_referenced": 0.73,
            "p_directly_studied": 0.04,
            "novelty_score": 0.89,
            "feasibility_score": 0.71,
            "impact_score": 0.85,
            "domain_a": "block_copolymer_mechanical",
            "domain_b": "high_temp_homopolymer"
        },
        {
            "id": "GAP-002",
            "description": (
                "Humidity-dependent degradation kinetics of "
                "UV-exposed polymer coatings"
            ),
            "strategy": "negative_space",
            "p_referenced": 0.68,
            "p_directly_studied": 0.09,
            "novelty_score": 0.74,
            "feasibility_score": 0.82,
            "impact_score": 0.63,
            "domain_a": "polymer_aging",
            "domain_b": None
        },
        {
            "id": "GAP-003",
            "description": (
                "Long-term creep behavior of nanocomposite-"
                "reinforced polyimides under cyclic thermal loading"
            ),
            "strategy": "temporal_trend",
            "p_referenced": 0.55,
            "p_directly_studied": 0.12,
            "novelty_score": 0.66,
            "feasibility_score": 0.58,
            "impact_score": 0.72,
            "domain_a": "nanocomposite_reinforcement",
            "domain_b": "processing_property"
        }
    ],
    "methodology_used": ["negative_space", "cross_domain", "temporal_trend"],
    "_source": "13.6 Knowledge Gap Identification"
}

# ─── Dataset 9: Experiment Rounds (§13.8) ───
# Source: Closed-loop feedback, pp. 386–387
# Demonstrates error reduction 12% → 8% → 5%
MOCK_EXPERIMENT_ROUNDS = [
    {
        "round": 1,
        "hypothesis_id": "H1-aromatic-block",
        "predicted": {"tg_celsius": 338, "tensile_mpa": 95, "elongation_pct": 13.2},
        "measured":  {"tg_celsius": 355, "tensile_mpa": 108, "elongation_pct": 15.8},
        "avg_error_pct": 12.0
    },
    {
        "round": 2,
        "hypothesis_id": "H1-aromatic-block-v2",
        "predicted": {"tg_celsius": 348, "tensile_mpa": 102, "elongation_pct": 15.1},
        "measured":  {"tg_celsius": 352, "tensile_mpa": 107, "elongation_pct": 16.0},
        "avg_error_pct": 8.0
    },
    {
        "round": 3,
        "hypothesis_id": "H1-aromatic-block-v3",
        "predicted": {"tg_celsius": 353, "tensile_mpa": 106, "elongation_pct": 15.7},
        "measured":  {"tg_celsius": 355, "tensile_mpa": 108, "elongation_pct": 15.8},
        "avg_error_pct": 5.0
    }
]

log_success(f"9 mock datasets loaded: {len(MOCK_PATIENT_VITALS)} vital fields, "
            f"{len(MOCK_DIAGNOSES)} diagnoses, {len(MOCK_DRUG_DB)} drug interactions, "
            f"{len(MOCK_FHIR_BUNDLE['entry'])} FHIR resources, "
            f"{len(MOCK_PAPER_CORPUS)} papers, {len(MOCK_GAP_REPORT['gaps'])} gaps, "
            f"{len(MOCK_EXPERIMENT_ROUNDS)} experiment rounds.")


[SUCCESS] 9 mock datasets loaded: 11 vital fields, 5 diagnoses, 2 drug interactions, 4 FHIR resources, 15 papers, 3 gaps, 3 experiment rounds.


In [8]:
# Cell 1.5 — @graceful_fallback Decorator
# Ref: Cross-cutting resilience pattern — wraps every agent I/O method
# Author: Imran Ahmad

def graceful_fallback(fallback_value=None, section_ref="Unknown"):
    """
    Catches exceptions, logs in RED, returns fallback value.
    Ensures the notebook never terminates on a single tool failure.

    Ref: Resilience Layer Specification
    Author: Imran Ahmad

    Usage:
        @graceful_fallback(
            fallback_value=lambda: MOCK_DRUG_DB,
            section_ref="13.1 Drug Interaction Check"
        )
        def check_drug_interactions(medications):
            return drug_db_client.query(medications)
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            try:
                result = func(*args, **kwargs)
                log_success(
                    f"Step complete. {func.__name__} returned valid output."
                )
                return result
            except Exception as e:
                log_error(
                    f"{type(e).__name__}: {e}. "
                    f"Falling back to mock logic for {section_ref}."
                )
                if callable(fallback_value):
                    return fallback_value()
                return fallback_value
        return wrapper
    return decorator

# ─── Quick test: deliberate error triggers red fallback ───
@graceful_fallback(fallback_value=lambda: {"status": "fallback_ok"}, section_ref="Test")
def _test_decorator():
    raise ConnectionError("Simulated API failure")

_test_result = _test_decorator()
assert _test_result == {"status": "fallback_ok"}, "Decorator fallback failed"
log_success(f"@graceful_fallback decorator verified: {_test_result}")


[HANDLED ERROR] ConnectionError: Simulated API failure. Falling back to mock logic for Test.
[SUCCESS] @graceful_fallback decorator verified: {'status': 'fallback_ok'}


In [9]:
# Cell 1.4 — Mock Stub Classes for All Architecture Dependencies
# Ref: §13.1–13.8 — each stub mirrors the book's constructor interface
# Author: Imran Ahmad
#
# CRITICAL: Each stub:
#   - Accepts the same constructor args as the book code
#   - Returns domain-appropriate mock data from the registry
#   - Logs via log_info with [SIMULATION] prefix and section reference
#   - Never raises exceptions (all errors caught internally)

# ═══════════════════════════════════════════════════════════════
# Healthcare Agent Stubs (§13.1–13.4)
# ═══════════════════════════════════════════════════════════════

class DrugInteractionDB:
    """Ref: §13.1, p. 365 — drug interaction database stub."""
    def __init__(self, sources=None, update_frequency="daily"):
        self.sources = sources or ["drugbank", "rxnorm", "fda_labels"]
        self.update_frequency = update_frequency
        log_info(f"[SIMULATION] DrugInteractionDB initialized | sources: {self.sources}")

    def check_interactions(self, medications):
        log_info("[SIMULATION] DrugInteractionDB.check_interactions() | §13.1")
        return MOCK_DRUG_DB


class ClinicalGuidelineEngine:
    """Ref: §13.1, p. 365 — clinical guideline retrieval stub."""
    def __init__(self, sources=None, version_tracking=True):
        self.sources = sources or ["nice", "who", "aha", "idsa"]
        self.version_tracking = version_tracking
        log_info(f"[SIMULATION] ClinicalGuidelineEngine initialized | sources: {self.sources}")

    def search(self, clinical_context):
        log_info("[SIMULATION] ClinicalGuidelineEngine.search() | §13.1")
        return [{
            "guideline": "Surviving Sepsis Campaign 2024",
            "recommendation": "Broad-spectrum antibiotics within 1 hour",
            "evidence_grade": "1A",
            "origin": "idsa",
            "source_version": "2024.1",
            "source_reliability_score": 0.96,
            "provenance": {
                "source": "idsa", "version": "2024.1",
                "retrieved_at": datetime.now(timezone.utc).isoformat(),
                "confidence": 0.96
            }
        }]


class DiseaseOntology:
    """Ref: §13.1, p. 365 — disease ontology stub."""
    def __init__(self, base="snomed_ct", extensions=None):
        self.base = base
        self.extensions = extensions or ["icd10", "orphanet"]
        log_info(f"[SIMULATION] DiseaseOntology initialized | base: {self.base}")

    def match_symptoms(self, symptom_profile):
        log_info("[SIMULATION] DiseaseOntology.match_symptoms() | §13.1")
        return [{
            "condition": "sepsis",
            "snomed_code": "91302008",
            "match_score": 0.87,
            "origin": "snomed_ct",
            "source_version": "2026.03",
            "source_reliability_score": 0.94,
            "provenance": {
                "source": "snomed_ct", "version": "2026.03",
                "retrieved_at": datetime.now(timezone.utc).isoformat(),
                "confidence": 0.94
            }
        }]


class MedicalLiteratureIndex:
    """Ref: §13.1, p. 366 — literature index stub."""
    def __init__(self, sources=None, embedding_model="biomedical-bert"):
        self.sources = sources or ["pubmed", "cochrane", "uptodate"]
        self.embedding_model = embedding_model
        log_info(f"[SIMULATION] MedicalLiteratureIndex initialized | model: {self.embedding_model}")

    def search(self, query, top_k=5):
        log_info("[SIMULATION] MedicalLiteratureIndex.search() | §13.1")
        return [{
            "title": "Early Recognition and Management of Sepsis",
            "source": "cochrane",
            "year": 2024,
            "relevance_score": 0.91,
            "origin": "cochrane",
            "source_version": "2024.Q4",
            "source_reliability_score": 0.98
        }]


class RateLimiter:
    """Ref: §13.5, p. 376 — rate limiter for API clients."""
    def __init__(self, max_per_second=None, min_interval_seconds=None):
        self.max_per_second = max_per_second
        self.min_interval_seconds = min_interval_seconds


# ─── FHIR Adapters (§13.2) ───

class FHIRResourceValidator:
    """Ref: §13.2, p. 367 — FHIR resource validation stub."""
    def __init__(self, profile="us-core-6.0"):
        self.profile = profile
        log_info(f"[SIMULATION] FHIRResourceValidator initialized | profile: {self.profile}")

    def validate(self, resource):
        log_info(f"[SIMULATION] Validating FHIR resource: {resource.get('resourceType', 'unknown')}")
        return type("ValidationResult", (), {"is_valid": True, "errors": []})()


class HL7v2ToFHIRAdapter:
    """Ref: §13.2, p. 367 — HL7v2 to FHIR adapter stub."""
    def transform(self, raw_data):
        log_info("[SIMULATION] HL7v2ToFHIRAdapter.transform() | §13.2")
        return [raw_data] if isinstance(raw_data, dict) else raw_data


class PassthroughAdapter:
    """Ref: §13.2, p. 367 — passthrough for native FHIR R4."""
    def transform(self, raw_data):
        return [raw_data] if isinstance(raw_data, dict) else raw_data


class CSVLabResultAdapter:
    """Ref: §13.2, p. 367 — CSV lab result adapter stub."""
    def transform(self, raw_data):
        log_info("[SIMULATION] CSVLabResultAdapter.transform() | §13.2")
        return [raw_data] if isinstance(raw_data, dict) else raw_data


class EpicFHIRAdapter:
    """Ref: §13.2, p. 367 — Epic EHR adapter stub."""
    def transform(self, raw_data):
        log_info("[SIMULATION] EpicFHIRAdapter.transform() | §13.2")
        return [raw_data] if isinstance(raw_data, dict) else raw_data


class CernerFHIRAdapter:
    """Ref: §13.2, p. 367 — Cerner EHR adapter stub."""
    def transform(self, raw_data):
        log_info("[SIMULATION] CernerFHIRAdapter.transform() | §13.2")
        return [raw_data] if isinstance(raw_data, dict) else raw_data


# ─── Patient Data Pipeline Sub-Agents (§13.2) ───

class HeartRateProcessor:
    """Ref: §13.2, p. 368 — heart rate processing stub."""
    def process(self, data):
        return {"heart_rate_bpm": data.get("heart_rate_bpm", 72), "status": "analyzed"}


class BloodPressureProcessor:
    """Ref: §13.2, p. 368."""
    def process(self, data):
        return {"systolic": data.get("blood_pressure_systolic", 120),
                "diastolic": data.get("blood_pressure_diastolic", 80), "status": "analyzed"}


class BloodGlucoseProcessor:
    """Ref: §13.2, p. 368."""
    def process(self, data):
        return {"glucose_mg_dl": 105, "status": "analyzed"}


class SpO2Processor:
    """Ref: §13.2, p. 368."""
    def process(self, data):
        return {"spo2_percent": data.get("spo2_percent", 98), "status": "analyzed"}


class ActivityProcessor:
    """Ref: §13.2, p. 368."""
    def process(self, data):
        return {"activity_level": "low", "status": "analyzed"}


class BiometricAnalyzer:
    """Ref: §13.2, p. 368 — multi-sensor biometric analysis stub."""
    def __init__(self, processors=None):
        self.processors = processors or {}
        log_info("[SIMULATION] BiometricAnalyzer initialized | §13.2")

    def analyze(self, data):
        log_info("[SIMULATION] BiometricAnalyzer.analyze() | §13.2")
        if isinstance(data, dict):
            return {
                "temperature_c": data.get("temperature_c", 37.0),
                "heart_rate_bpm": data.get("heart_rate_bpm", 72),
                "blood_pressure": f"{data.get('blood_pressure_systolic', 120)}/{data.get('blood_pressure_diastolic', 80)}",
                "spo2_percent": data.get("spo2_percent", 98),
                "map_mmhg": data.get("map_mmhg", 80),
                "analysis_timestamp": datetime.now(timezone.utc).isoformat(),
                "anomalies_detected": ["tachycardia", "hypotension", "fever"]
            }
        return {"status": "analyzed", "anomalies_detected": []}


class SymptomInterpreter:
    """Ref: §13.2, p. 368 — NLP-based symptom interpretation stub."""
    def __init__(self, nlp_model="clinical-bert", symptom_ontology="medra"):
        self.nlp_model = nlp_model
        self.symptom_ontology = symptom_ontology
        log_info(f"[SIMULATION] SymptomInterpreter initialized | model: {self.nlp_model}")

    def interpret(self, symptoms, patient_context=None):
        log_info("[SIMULATION] SymptomInterpreter.interpret() | §13.2")
        return {
            "primary_symptoms": ["fever_with_rigors", "tachycardia", "hypotension"],
            "flagged_conditions": ["sepsis", "urinary_tract_infection"],
            "severity": "HIGH",
            "confidence": 0.84,
            "symptom_profile": symptoms if isinstance(symptoms, list) else [symptoms]
        }


class PatientHistoryAgent:
    """Ref: §13.2, p. 368 — episodic patient history stub."""
    def __init__(self, memory_type="episodic", retention_policy="hipaa_compliant"):
        self.memory_type = memory_type
        self.retention_policy = retention_policy
        log_info(f"[SIMULATION] PatientHistoryAgent initialized | memory: {self.memory_type}")

    def retrieve(self, patient_id, relevance_window="36_months", priority_conditions=None):
        log_info(f"[SIMULATION] PatientHistoryAgent.retrieve({patient_id}) | §13.2")
        return {
            "patient_id": patient_id,
            "conditions": ["type_2_diabetes", "hypertension"],
            "medications": ["metformin", "lisinopril"],
            "recent_encounters": [
                {"date": "2026-02-15", "type": "routine_checkup", "notes": "Stable A1c 6.8%"},
                {"date": "2026-01-10", "type": "lab_work", "notes": "Renal function normal"}
            ],
            "relevance_window": relevance_window
        }


# ─── Diagnostic Coordinator Sub-Components (§13.3) ───

class DifferentialGenerator:
    """Ref: §13.3, p. 370 — differential diagnosis generator stub."""
    def __init__(self):
        log_info("[SIMULATION] DifferentialGenerator initialized | §13.3")
        self._trace = []

    def generate_differentials(self, vitals, symptoms, history):
        log_info("[SIMULATION] DifferentialGenerator.generate_differentials() | §13.3")
        differentials = [
            {"diagnosis": "urosepsis", "raw_score": 0.61},
            {"diagnosis": "pneumonia_sepsis", "raw_score": 0.21},
            {"diagnosis": "biliary_sepsis", "raw_score": 0.11},
            {"diagnosis": "viral_syndrome", "raw_score": 0.04},
            {"diagnosis": "dehydration", "raw_score": 0.03}
        ]
        self._trace = [
            "Step 1: Vitals pattern matched to infectious syndrome",
            "Step 2: Symptom profile weighted toward urinary source",
            "Step 3: History of diabetes increases UTI susceptibility",
            "Step 4: Hemodynamic instability escalates to sepsis consideration"
        ]
        return differentials

    def get_trace(self):
        return self._trace


class ClinicalExplainer:
    """Ref: §13.3, p. 374 — audience-adapted explanation generator stub."""
    def __init__(self):
        log_info("[SIMULATION] ClinicalExplainer initialized | §13.3")

    def generate(self, scored, audience="clinician", evidence_sources=None):
        log_info(f"[SIMULATION] ClinicalExplainer.generate(audience={audience}) | §13.3")
        if audience == "clinician":
            return (
                "SAFETY ALERT — Escalation required. Primary concern: Sepsis "
                "(confidence above escalation threshold: 0.82). Key findings: "
                "temperature 38.9°C with rigors (SHAP contribution: 0.34), "
                "heart rate 118 bpm (0.27), WBC 18.4 with left shift (0.22), "
                "MAP trending toward 65 mmHg (0.17). SOFA score estimate: 4. "
                "Differential: Urosepsis (0.61), pneumonia-source sepsis (0.21), "
                "biliary source (0.11). Immediate action: Blood cultures x2, lactate, "
                "broad-spectrum antibiotics within 1 hour per Surviving Sepsis "
                "Campaign protocol. Attending notification triggered."
            )
        else:
            return (
                "Your temperature, heart rate, and blood test results together "
                "suggest your body may be fighting a serious infection. Your care "
                "team has been notified and will be with you shortly. They may start "
                "antibiotics and run additional tests to identify the source of the "
                "infection. This is a precaution to make sure you receive the right "
                "treatment quickly."
            )


class ConfidenceAwareAgent:
    """Ref: §13.3, p. 370 — Platt-calibrated confidence scorer stub."""
    def __init__(self, n_hypotheses=5, calibration_method="platt_scaling"):
        self.n_hypotheses = n_hypotheses
        self.calibration_method = calibration_method
        self._calibration_report = {}
        log_info(f"[SIMULATION] ConfidenceAwareAgent initialized | method: {self.calibration_method}")

    def score_differentials(self, differentials, evidence=None):
        log_info("[SIMULATION] ConfidenceAwareAgent.score_differentials() | §13.3")
        scored = []
        for d in differentials:
            calibrated = min(d["raw_score"] * 1.15, 0.99)
            scored.append({
                **d,
                "calibrated_confidence": round(calibrated, 3),
                "calibration_method": self.calibration_method
            })
        self._calibration_report = {
            "method": self.calibration_method,
            "brier_score": 0.12,
            "reliability": 0.03,
            "resolution": 0.15,
            "n_hypotheses": len(scored)
        }
        return scored

    def get_calibration_report(self):
        return self._calibration_report

    def communicate_uncertainty(self, scored):
        top = scored[0] if scored else {}
        return {
            "top_diagnosis": top.get("diagnosis", "unknown"),
            "confidence": top.get("calibrated_confidence", 0.0),
            "interpretation": "HIGH" if top.get("calibrated_confidence", 0) > 0.5 else "MODERATE",
            "calibration_method": self.calibration_method
        }


class ClinicalMemorySystem:
    """Ref: §13.3, p. 370 — episodic + semantic clinical memory stub."""
    def __init__(self):
        self._store = {}
        log_info("[SIMULATION] ClinicalMemorySystem initialized | §13.3")

    def retrieve_episodic(self, patient_id):
        log_info(f"[SIMULATION] ClinicalMemorySystem.retrieve_episodic({patient_id}) | §13.3")
        return self._store.get(patient_id, {
            "previous_diagnoses": [],
            "treatment_history": [],
            "note": "No prior episodic memory for this patient"
        })

    def store_episodic(self, patient_id, data):
        log_info(f"[SIMULATION] ClinicalMemorySystem.store_episodic({patient_id}) | §13.3")
        self._store[patient_id] = data


class SafetyMonitor:
    """Ref: §13.3, p. 370 — safety escalation with 0.15 threshold (pp. 373–374)."""
    def __init__(self, escalation_threshold=0.15, critical_conditions=None):
        self.escalation_threshold = escalation_threshold
        self.critical_conditions = critical_conditions or [
            "myocardial_infarction", "pulmonary_embolism", "sepsis", "stroke"
        ]
        log_info(
            f"[SIMULATION] SafetyMonitor initialized | threshold: {self.escalation_threshold} | "
            f"critical conditions: {self.critical_conditions}"
        )

    def evaluate(self, scored_differentials):
        log_info("[SIMULATION] SafetyMonitor.evaluate() | §13.3")
        requires_escalation = False
        alerts = []
        for d in scored_differentials:
            diag = d.get("diagnosis", "")
            conf = d.get("calibrated_confidence", 0)
            # Check if any diagnosis matches or contains a critical condition keyword
            for critical in self.critical_conditions:
                if critical in diag or "sepsis" in diag:
                    if conf >= self.escalation_threshold:
                        requires_escalation = True
                        alerts.append({
                            "condition": diag,
                            "confidence": conf,
                            "action": "IMMEDIATE_ESCALATION"
                        })
        return type("SafetyResult", (), {
            "requires_escalation": requires_escalation,
            "alerts": alerts,
            "threshold_used": self.escalation_threshold
        })()

    def trigger_alert(self, patient_id, safety_result):
        log_warning(
            f"SAFETY ESCALATION for patient {patient_id}: "
            f"{len(safety_result.alerts)} critical alert(s) triggered"
        )
        for alert in safety_result.alerts:
            log_warning(
                f"  → {alert['condition']} (confidence: {alert['confidence']:.2f}) — "
                f"{alert['action']}"
            )


class AuditTrailGenerator:
    """Ref: §13.3, p. 373 — immutable audit trail stub."""
    def __init__(self):
        self._trail = []
        log_info("[SIMULATION] AuditTrailGenerator initialized | §13.3")

    def record(self, event):
        entry_id = f"AUDIT-{len(self._trail)+1:04d}"
        entry = {
            "id": entry_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "event": event if isinstance(event, dict) else str(event),
            "hash": hashlib.sha256(str(event).encode()).hexdigest()[:16]
        }
        self._trail.append(entry)
        log_info(f"[SIMULATION] Audit trail recorded: {entry_id} | hash: {entry['hash']}")
        return entry_id

    def get_last_entry_id(self):
        return self._trail[-1]["id"] if self._trail else None

    def get_trail(self):
        return self._trail


# ═══════════════════════════════════════════════════════════════
# Scientific Discovery Agent Stubs (§13.5–13.8)
# ═══════════════════════════════════════════════════════════════

class PubMedClient:
    """Ref: §13.5, p. 376 — PubMed API client stub."""
    def __init__(self, api_key=None, rate_limit=None):
        self.api_key = api_key
        self.rate_limit = rate_limit
        log_info("[SIMULATION] PubMedClient initialized | §13.5")

    async def search(self, query, max_results=500):
        log_info(f"[SIMULATION] PubMedClient.search('{query[:40]}...') | §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "pubmed"]


class ArxivClient:
    """Ref: §13.5, p. 377 — arXiv API client stub."""
    def __init__(self, rate_limit=None):
        self.rate_limit = rate_limit
        log_info("[SIMULATION] ArxivClient initialized | §13.5")

    async def search(self, query, max_results=500):
        log_info(f"[SIMULATION] ArxivClient.search('{query[:40]}...') | §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "arxiv"]


class ScopusClient:
    """Ref: §13.5, p. 377 — Scopus API client stub."""
    def __init__(self, api_key=None, monthly_budget=20000):
        self.api_key = api_key
        self.monthly_budget = monthly_budget
        log_info("[SIMULATION] ScopusClient initialized | §13.5")

    async def search(self, query, max_results=500):
        log_info(f"[SIMULATION] ScopusClient.search('{query[:40]}...') | §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "scopus"]


class IEEEXploreClient:
    """Ref: §13.5, p. 377 — IEEE Xplore API client stub."""
    def __init__(self, api_key=None, monthly_budget=10000):
        self.api_key = api_key
        self.monthly_budget = monthly_budget
        log_info("[SIMULATION] IEEEXploreClient initialized | §13.5")

    async def search(self, query, max_results=500):
        log_info(f"[SIMULATION] IEEEXploreClient.search('{query[:40]}...') | §13.5")
        return [p for p in MOCK_PAPER_CORPUS if p["source_db"] == "ieee"]


class ResultCache:
    """Ref: §13.5, p. 377 — result cache stub."""
    def __init__(self, backend="redis", ttl_hours=168):
        self.backend = backend
        self.ttl_hours = ttl_hours
        self._cache = {}
        log_info(f"[SIMULATION] ResultCache initialized | backend: {self.backend}")

    def get(self, query):
        cached = self._cache.get(query)
        if cached:
            log_info(f"[SIMULATION] Cache HIT for query: '{query[:30]}...'")
            return type("CacheEntry", (), {"results": cached, "is_stale": False})()
        return None

    def set(self, query, results):
        self._cache[query] = results
        log_info(f"[SIMULATION] Cache SET for query: '{query[:30]}...' ({len(results)} results)")


class PaperDeduplicator:
    """Ref: §13.5, p. 377 — paper deduplication stub."""
    def __init__(self, match_on=None, similarity_threshold=0.95):
        self.match_on = match_on or ["doi", "title_similarity"]
        self.similarity_threshold = similarity_threshold
        log_info("[SIMULATION] PaperDeduplicator initialized | §13.5")

    def deduplicate(self, papers):
        seen_dois = set()
        unique = []
        for paper in papers:
            doi = paper.get("doi", "")
            if doi not in seen_dois:
                seen_dois.add(doi)
                unique.append(paper)
        removed = len(papers) - len(unique)
        if removed:
            log_info(f"[SIMULATION] Deduplication: removed {removed} duplicate(s)")
        return unique


# ─── Knowledge Gap Sub-Components (§13.6) ───

class CitationGraphAnalyzer:
    """Ref: §13.6, p. 380 — citation graph analysis stub."""
    def __init__(self):
        log_info("[SIMULATION] CitationGraphAnalyzer initialized | §13.6")

    def analyze(self, corpus):
        log_info("[SIMULATION] CitationGraphAnalyzer.analyze() | §13.6")
        return {"total_citations": sum(p.get("citations", 0) for p in corpus),
                "avg_citations": np.mean([p.get("citations", 0) for p in corpus])}


class ResearchDomainMapper:
    """Ref: §13.6, p. 380 — research domain boundary detection stub."""
    def __init__(self):
        log_info("[SIMULATION] ResearchDomainMapper initialized | §13.6")

    def find_unexplored_intersections(self, clusters, min_relevance=0.7):
        log_info("[SIMULATION] ResearchDomainMapper.find_unexplored_intersections() | §13.6")
        return [MOCK_GAP_REPORT["gaps"][0]]  # Cross-domain gap


class TemporalTrendTracker:
    """Ref: §13.6, p. 380 — publication trend tracking stub."""
    def __init__(self):
        log_info("[SIMULATION] TemporalTrendTracker initialized | §13.6")

    def find_abandoned_questions(self, corpus, declining_since="3_years",
                                  citation_status="still_cited"):
        log_info("[SIMULATION] TemporalTrendTracker.find_abandoned_questions() | §13.6")
        return [MOCK_GAP_REPORT["gaps"][2]]  # Temporal trend gap


# ─── Hypothesis Generation Sub-Components (§13.7) ───

class TheoreticalFrameworkRetriever:
    """Ref: §13.7, p. 383 — theoretical framework retrieval stub."""
    def __init__(self):
        log_info("[SIMULATION] TheoreticalFrameworkRetriever initialized | §13.7")

    def retrieve(self, domain, related_concepts):
        log_info(f"[SIMULATION] TheoreticalFrameworkRetriever.retrieve({domain}) | §13.7")
        return [{
            "framework": "Structure-Property Relationships in Block Copolymers",
            "key_principles": [
                "Segment length controls phase separation morphology",
                "Aromatic backbone rigidity correlates with Tg",
                "Block ratio determines mechanical flexibility"
            ],
            "relevance_score": 0.88
        }]


class ScientificReasoningEngine:
    """Ref: §13.7, p. 383 — abductive reasoning engine stub."""
    def __init__(self, model="scientific-llm", reasoning_modes=None):
        self.model = model
        self.reasoning_modes = reasoning_modes or ["analogical", "deductive", "abductive"]
        log_info(f"[SIMULATION] ScientificReasoningEngine initialized | modes: {self.reasoning_modes}")

    def reason(self, gap, frameworks, reasoning_type="abductive", constraints=None):
        log_info(f"[SIMULATION] ScientificReasoningEngine.reason(type={reasoning_type}) | §13.7")
        return [
            {
                "id": "H1",
                "statement": (
                    "Alternating aromatic dianhydride-diamine block copolymer with "
                    "segment length 15-20 repeat units will achieve Tg > 350°C "
                    "with elongation at break > 15%."
                ),
                "mechanism": "Controlled block length enables phase separation maintaining both rigidity and flexibility",
                "reasoning_type": reasoning_type,
                "gap_id": gap.get("id", "unknown") if isinstance(gap, dict) else "unknown"
            },
            {
                "id": "H2",
                "statement": (
                    "Incorporating flexible ether linkages at block junctions "
                    "will improve elongation by 40% with < 5% Tg reduction."
                ),
                "mechanism": "Ether linkages provide rotational freedom at junction points",
                "reasoning_type": reasoning_type,
                "gap_id": gap.get("id", "unknown") if isinstance(gap, dict) else "unknown"
            },
            {
                "id": "H3",
                "statement": (
                    "Nano-scale phase separation in aromatic block copolymers "
                    "creates dual-phase morphology combining crystalline thermal "
                    "stability with amorphous flexibility."
                ),
                "mechanism": "Block architecture enables microphase separation",
                "reasoning_type": reasoning_type,
                "gap_id": gap.get("id", "unknown") if isinstance(gap, dict) else "unknown"
            }
        ]


class HypothesisEvaluator:
    """Ref: §13.7, p. 383 — hypothesis scoring stub."""
    def __init__(self):
        log_info("[SIMULATION] HypothesisEvaluator initialized | §13.7")

    def evaluate(self, hypothesis, criteria=None):
        log_info(f"[SIMULATION] HypothesisEvaluator.evaluate({hypothesis.get('id', '?')}) | §13.7")
        return {
            "internal_consistency": 0.85,
            "novelty_vs_existing": 0.78,
            "testability": 0.82,
            "potential_impact": 0.76,
            "overall_score": 0.80
        }


# ─── Experiment Tracking Sub-Components (§13.8) ───

class PredictionDatabase:
    """Ref: §13.8, p. 386 — prediction store stub."""
    def __init__(self):
        self._store = {}
        log_info("[SIMULATION] PredictionDatabase initialized | §13.8")

    def store(self, hypothesis_id, predictions):
        self._store[hypothesis_id] = {
            "hypothesis_id": hypothesis_id,
            "predicted_properties": predictions,
            "timestamp": datetime.now(timezone.utc).isoformat()
        }

    def get(self, hypothesis_id):
        return self._store.get(hypothesis_id, {
            "predicted_properties": {"tg_celsius": 338, "tensile_mpa": 95, "elongation_pct": 13.2}
        })


class ExperimentalResultDatabase:
    """Ref: §13.8, p. 386 — experimental result store stub."""
    def __init__(self):
        self._results = []
        log_info("[SIMULATION] ExperimentalResultDatabase initialized | §13.8")

    def insert(self, record):
        self._results.append(record)
        log_info(f"[SIMULATION] Experiment result recorded: {record.get('hypothesis_id', '?')} | §13.8")

    def get_all(self):
        return self._results


class FeedbackEngine:
    """Ref: §13.8, p. 386 — model update feedback engine stub."""
    def __init__(self):
        self._update_count = 0
        log_info("[SIMULATION] FeedbackEngine initialized | §13.8")

    def update_models(self, hypothesis_id, errors):
        self._update_count += 1
        avg_error = np.mean([v["error_pct"] for v in errors.values()]) if errors else 0
        log_info(
            f"[SIMULATION] FeedbackEngine.update_models() | iteration {self._update_count} | "
            f"avg_error: {avg_error:.1f}% | §13.8"
        )


# ─── Helper dataclass-like containers ───

class UnsupportedFormatError(Exception):
    """Raised when a data format has no registered adapter."""
    pass


class InferenceEvent(dict):
    """Container for audit trail inference events. Ref: §13.3, p. 373."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)


# ─── Summary ───
_stub_count = sum(1 for name, obj in list(locals().items())
                  if isinstance(obj, type) and name[0].isupper()
                  and name not in ("Dict", "List", "Optional", "Any",
                                   "Callable", "Tuple"))
log_success(f"Mock stub classes loaded: {_stub_count} classes ready for Simulation Mode.")


[SUCCESS] Mock stub classes loaded: 45 classes ready for Simulation Mode.


In [10]:
# Cell 1.6 — Additional Mock Fallback Objects
# Ref: Decorator Application Map — fallback values for @graceful_fallback
# Author: Imran Ahmad

# Pre-built clinical context for PatientDataPipeline fallback
MOCK_CLINICAL_CONTEXT = {
    "patient_id": "SIM-PT-00421",
    "vitals": {
        "temperature_c": 38.9,
        "heart_rate_bpm": 118,
        "blood_pressure": "92/58",
        "spo2_percent": 94,
        "map_mmhg": 65,
        "anomalies_detected": ["tachycardia", "hypotension", "fever"]
    },
    "symptoms": {
        "primary_symptoms": ["fever_with_rigors", "tachycardia", "hypotension"],
        "flagged_conditions": ["sepsis", "urinary_tract_infection"],
        "severity": "HIGH"
    },
    "history": {
        "conditions": ["type_2_diabetes", "hypertension"],
        "medications": ["metformin", "lisinopril"]
    },
    "timeline": [
        {"time": "2026-03-30T06:00:00Z", "event": "Fever onset (38.2°C)"},
        {"time": "2026-03-30T08:00:00Z", "event": "Rigors reported"},
        {"time": "2026-03-30T10:30:00Z", "event": "ED presentation — vitals recorded"}
    ],
    "data_quality": "complete",
    "_source": "13.2 Patient Data Pipeline fallback"
}

# Pre-built diagnostic report for DiagnosticCoordinator fallback
MOCK_DIAGNOSTIC_REPORT = {
    "differentials": [
        {"diagnosis": "urosepsis", "calibrated_confidence": 0.70},
        {"diagnosis": "pneumonia_sepsis", "calibrated_confidence": 0.24},
        {"diagnosis": "biliary_sepsis", "calibrated_confidence": 0.13}
    ],
    "explanation": "Fallback diagnostic report — see §13.3 for full pipeline.",
    "confidence_summary": {"top_diagnosis": "urosepsis", "confidence": 0.70},
    "audit_trail": "AUDIT-FALLBACK",
    "_source": "13.3 DiagnosticCoordinator fallback"
}

# Pre-built hypotheses for HypothesisGenerator fallback
MOCK_HYPOTHESES = [
    {
        "id": "H1",
        "statement": "Alternating aromatic dianhydride-diamine block copolymer achieves Tg > 350°C with elongation > 15%",
        "consistency_score": 0.85,
        "testability_score": 0.82,
        "proposed_experiments": ["DSC thermal analysis", "Tensile testing per ASTM D638"]
    }
]

log_success("Fallback objects defined: MOCK_CLINICAL_CONTEXT, MOCK_DIAGNOSTIC_REPORT, MOCK_HYPOTHESES.")


[SUCCESS] Fallback objects defined: MOCK_CLINICAL_CONTEXT, MOCK_DIAGNOSTIC_REPORT, MOCK_HYPOTHESES.


In [11]:
# Cell 1.7 — Section 1 Validation
# Ref: Cross-cutting QA — verify all infrastructure is operational
# Author: Imran Ahmad

print("=" * 70)
print("  SECTION 1 VALIDATION — Simulation Infrastructure")
print("=" * 70)
print()

checks = {
    "MockLLM responds":          lambda: isinstance(llm.invoke("test", "diagnostic"), MockResponse),
    "6 context types":           lambda: len(llm._response_registry) == 6,
    "9 mock datasets":           lambda: all([
        MOCK_PATIENT_VITALS, MOCK_DIAGNOSES is not None, MOCK_PRIOR_BELIEF is not None,
        MOCK_LIKELIHOOD_SCORES, MOCK_DRUG_DB, MOCK_FHIR_BUNDLE,
        MOCK_DEPLOYMENT_METRICS, MOCK_PAPER_CORPUS, MOCK_GAP_REPORT,
        MOCK_EXPERIMENT_ROUNDS
    ]),
    "15 papers in corpus":       lambda: len(MOCK_PAPER_CORPUS) == 15,
    "3 gaps in report":          lambda: len(MOCK_GAP_REPORT["gaps"]) == 3,
    "3 experiment rounds":       lambda: len(MOCK_EXPERIMENT_ROUNDS) == 3,
    "@graceful_fallback works":  lambda: _test_result == {"status": "fallback_ok"},
    "SafetyMonitor threshold":   lambda: SafetyMonitor().escalation_threshold == 0.15,
    "Fallback objects defined":  lambda: all([MOCK_CLINICAL_CONTEXT, MOCK_DIAGNOSTIC_REPORT, MOCK_HYPOTHESES]),
}

all_passed = True
for name, check in checks.items():
    try:
        result = check()
        status = "PASS" if result else "FAIL"
        if not result:
            all_passed = False
    except Exception as e:
        status = f"ERROR: {e}"
        all_passed = False
    print(f"  {'✓' if status == 'PASS' else '✗'} {name}: {status}")

print()
if all_passed:
    log_success("All Section 1 checks passed. Simulation infrastructure is operational.")
else:
    log_error("Some checks failed. Review output above.")
print("=" * 70)


  SECTION 1 VALIDATION — Simulation Infrastructure

[INFO] [SIMULATION] MockLLM call #1 | context: diagnostic | source: 13.3 Clinical Decision Support
  ✓ MockLLM responds: PASS
  ✓ 6 context types: PASS
  ✓ 9 mock datasets: PASS
  ✓ 15 papers in corpus: PASS
  ✓ 3 gaps in report: PASS
  ✓ 3 experiment rounds: PASS
  ✓ @graceful_fallback works: PASS
[INFO] [SIMULATION] SafetyMonitor initialized | threshold: 0.15 | critical conditions: ['myocardial_infarction', 'pulmonary_embolism', 'sepsis', 'stroke']
  ✓ SafetyMonitor threshold: PASS
  ✓ Fallback objects defined: PASS

[SUCCESS] All Section 1 checks passed. Simulation infrastructure is operational.


## Section 2: Healthcare Intelligence Agent
*Ref: §13.1–13.4 (pp. 362–375)*

This section implements the four-layer Healthcare Intelligence architecture:
- **Data Ingestion Layer** — FHIR normalization, biometric analysis, symptom interpretation
- **Clinical Knowledge Layer** — Dual-memory knowledge base with provenance tracking
- **Reasoning & Decision Layer** — Bayesian belief updating, confidence calibration, safety escalation
- **Explanation & Delivery Layer** — Audience-adapted explanations (clinician vs. patient)

> *"The very first requirement in a hospital is that it should do the sick no harm."* — Florence Nightingale

#### Figure 13.1 — Healthcare Intelligence Agent Architecture *(Book p. 363)*

The architecture comprises four primary layers, each communicating through well-defined interfaces. This enables independent knowledge base updates without disrupting the reasoning pipeline, and allows explanation formats to be customized for different audiences without modifying diagnostic logic.

```
┌─────────────────────────────────────────────────────────────────────┐
│                      DATA INGESTION LAYER                          │
│  ┌─────────────────┐ ┌─────────────────┐ ┌─────────────────────┐   │
│  │ Biometric Agents │ │ Symptom Agents  │ │ EHR Connectors     │   │
│  │ (heart rate,     │ │ (NLP: clinical- │ │ (FHIR R4, HL7v2,  │   │
│  │  BP, SpO2, etc.) │ │  bert, MedDRA)  │ │  Epic, Cerner)    │   │
│  └────────┬─────────┘ └────────┬────────┘ └─────────┬──────────┘   │
│           └────────────────────┼────────────────────┘              │
└────────────────────────────────┼───────────────────────────────────┘
                                 ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    CLINICAL KNOWLEDGE LAYER                        │
│  ┌─────────────────┐ ┌─────────────────┐ ┌─────────────────────┐   │
│  │ Semantic Memory  │ │ Drug Interaction │ │ Clinical Guidelines │   │
│  │ (SNOMED-CT,     │ │ DB (DrugBank,   │ │ (NICE, WHO, AHA,   │   │
│  │  ICD-10)        │ │  RxNorm, FDA)   │ │  IDSA)             │   │
│  └────────┬─────────┘ └────────┬────────┘ └─────────┬──────────┘   │
│           └────────────────────┼────────────────────┘              │
└────────────────────────────────┼───────────────────────────────────┘
                                 ▼
┌─────────────────────────────────────────────────────────────────────┐
│                  REASONING & DECISION LAYER                        │
│  ┌─────────────────┐ ┌─────────────────┐ ┌─────────────────────┐   │
│  │  Differential   │ │   Bayesian      │ │  Safety Monitor    │   │
│  │  Diagnosis      │ │   Confidence    │ │  (threshold: 0.15, │   │
│  │  Generator      │ │   Scorer        │ │   4 critical cond.)│   │
│  └────────┬─────────┘ └────────┬────────┘ └─────────┬──────────┘   │
│           └────────────────────┼────────────────────┘              │
└────────────────────────────────┼───────────────────────────────────┘
                                 ▼
┌─────────────────────────────────────────────────────────────────────┐
│                EXPLANATION & DELIVERY LAYER                        │
│  ┌─────────────────┐ ┌─────────────────┐ ┌─────────────────────┐   │
│  │ Clinician Reports│ │ Patient         │ │ Immutable Audit    │   │
│  │ (SHAP values)   │ │ Summaries       │ │ Trail Generator    │   │
│  └─────────────────┘ └─────────────────┘ └─────────────────────┘   │
└─────────────────────────────────────────────────────────────────────┘
```

**Key design decision:** The feedback loops are narrow and explicit — explanation outcomes and clinician overrides can inform knowledge base updates without exposing the reasoning layer to raw feedback signals. This is a deliberate tradeoff: sacrificing some adaptability to preserve auditability.


### 2a: Bayesian Belief Update
*Ref: §13.1, p. 363 — POMDP belief-state update*

The agent maintains a probability distribution over candidate diagnoses and updates it as new observations arrive. The formal foundation:

$$P(D_i | O_1, \ldots, O_n) = \frac{P(O_1, \ldots, O_n | D_i) \cdot P(D_i)}{P(O_1, \ldots, O_n)}$$

The `update_belief()` function implements one cycle of this update: the prior belief is weighted by how well each diagnosis explains the new observation, then renormalized.

In [12]:
# Cell 2a — Bayesian Belief Update: update_belief()
# Ref: §13.1, p. 363 — exact implementation from book
# Author: Imran Ahmad

def update_belief(belief: np.ndarray,
                  observation: dict,
                  likelihood_model: dict) -> np.ndarray:
    """
    Bayesian belief update: P(s|o) ∝ P(o|s) * P(s).

    Advances the POMDP one step: the prior belief is weighted by how
    well each diagnosis explains the new observation, then renormalized.

    Ref: §13.1, p. 363 — Partially Observable Markov Decision Process
    Author: Imran Ahmad

    Args:
        belief: Prior probability distribution over diagnoses (numpy array).
        observation: Dict of observed clinical findings.
        likelihood_model: Dict mapping diagnosis names to likelihood scores.

    Returns:
        Posterior probability distribution (normalized numpy array).
    """
    likelihoods = np.array([
        likelihood_model[diag] if isinstance(likelihood_model[diag], (int, float))
        else likelihood_model[diag].score(observation)
        for diag in likelihood_model
    ])
    posterior = likelihoods * belief
    return posterior / posterior.sum()  # normalize to valid distribution


# ─── Demo: Run with mock data from §13.3 sepsis scenario ───
log_info("Running Bayesian belief update with sepsis scenario data...")
print()

print(f"  Prior belief:     {dict(zip(MOCK_DIAGNOSES, MOCK_PRIOR_BELIEF))}")
print(f"  Likelihood scores: {MOCK_LIKELIHOOD_SCORES}")
print()

posterior = update_belief(
    belief=MOCK_PRIOR_BELIEF,
    observation=MOCK_PATIENT_VITALS,
    likelihood_model=MOCK_LIKELIHOOD_SCORES
)

print("  Posterior belief (after observing sepsis-like vitals):")
print("  " + "-" * 55)
for diag, prior, post in zip(MOCK_DIAGNOSES, MOCK_PRIOR_BELIEF, posterior):
    shift = post - prior
    arrow = "▲" if shift > 0 else "▼"
    print(f"  {diag:<20s}  prior: {prior:.2f}  →  posterior: {post:.3f}  ({arrow} {abs(shift):.3f})")

print()
log_success(
    f"Bayesian update complete. Top diagnosis: {MOCK_DIAGNOSES[np.argmax(posterior)]} "
    f"(posterior: {posterior.max():.3f})"
)

# Verify posterior is a valid distribution
assert abs(posterior.sum() - 1.0) < 1e-10, "Posterior does not sum to 1.0"
log_success("Posterior is a valid probability distribution (sums to 1.0).")


[INFO] Running Bayesian belief update with sepsis scenario data...

  Prior belief:     {'urosepsis': np.float64(0.25), 'pneumonia_sepsis': np.float64(0.25), 'biliary_sepsis': np.float64(0.15), 'viral_syndrome': np.float64(0.2), 'dehydration': np.float64(0.15)}
  Likelihood scores: {'urosepsis': 0.82, 'pneumonia_sepsis': 0.65, 'biliary_sepsis': 0.45, 'viral_syndrome': 0.2, 'dehydration': 0.15}

  Posterior belief (after observing sepsis-like vitals):
  -------------------------------------------------------
  urosepsis             prior: 0.25  →  posterior: 0.412  (▲ 0.162)
  pneumonia_sepsis      prior: 0.25  →  posterior: 0.327  (▲ 0.077)
  biliary_sepsis        prior: 0.15  →  posterior: 0.136  (▼ 0.014)
  viral_syndrome        prior: 0.20  →  posterior: 0.080  (▼ 0.120)
  dehydration           prior: 0.15  →  posterior: 0.045  (▼ 0.105)

[SUCCESS] Bayesian update complete. Top diagnosis: urosepsis (posterior: 0.412)
[SUCCESS] Posterior is a valid probability distribution (sums to 1

### 2b: Clinical Knowledge Base
*Ref: §13.1, pp. 365–367 — Dual-memory architecture with provenance tracking*

The knowledge base integrates four authoritative sources (drug interaction DB, clinical guidelines, disease ontology, medical literature) into a unified retrieval interface. Every result carries provenance metadata for auditability: source, version, retrieval timestamp, and confidence score.

When conflicting guidelines are detected, the agent flags the conflict and presents both recommendations rather than silently choosing one.

In [13]:
# Cell 2b — ClinicalKnowledgeBase with Provenance Tracking
# Ref: §13.1, pp. 365–367 — dual-memory knowledge integration
# Author: Imran Ahmad

class ClinicalKnowledgeBase:
    """
    Integrates multiple medical knowledge sources into a unified
    retrieval interface with source tracking and currency validation.

    Ref: §13.1, pp. 365–367
    Author: Imran Ahmad
    """

    def __init__(self):
        self.drug_database = DrugInteractionDB(
            sources=["drugbank", "rxnorm", "fda_labels"],
            update_frequency="daily"
        )
        self.guideline_engine = ClinicalGuidelineEngine(
            sources=["nice", "who", "aha", "idsa"],
            version_tracking=True
        )
        self.disease_ontology = DiseaseOntology(
            base="snomed_ct",
            extensions=["icd10", "orphanet"]
        )
        self.literature_index = MedicalLiteratureIndex(
            sources=["pubmed", "cochrane", "uptodate"],
            embedding_model="biomedical-bert"
        )

    @graceful_fallback(fallback_value=lambda: MOCK_DRUG_DB, section_ref="13.1")
    def query(self, clinical_context: dict, query_type: str = "diagnostic") -> list:
        """
        Retrieve relevant clinical knowledge with provenance tracking.

        Ref: §13.1, p. 366 — query method with provenance metadata
        Author: Imran Ahmad
        """
        results = []

        if query_type in ["diagnostic", "treatment"]:
            results.extend(
                self.guideline_engine.search(clinical_context)
            )
            medications = clinical_context.get("medications", [])
            if medications:
                results.extend(
                    self.drug_database.check_interactions(medications)
                )

        if query_type == "differential":
            symptom_profile = clinical_context.get("symptom_profile", [])
            results.extend(
                self.disease_ontology.match_symptoms(symptom_profile)
            )

        # Attach provenance metadata to every result (§13.1, p. 366)
        for result in results:
            if isinstance(result, dict) and "provenance" not in result:
                result["provenance"] = {
                    "source": result.get("origin", "unknown"),
                    "version": result.get("source_version", "unknown"),
                    "retrieved_at": datetime.now(timezone.utc).isoformat(),
                    "confidence": result.get("source_reliability_score", 0.0)
                }

        return results


# ─── Demo: Query with mock clinical context ───
log_info("Initializing ClinicalKnowledgeBase...")
kb = ClinicalKnowledgeBase()

print()
log_info("Querying for diagnostic guidance (sepsis scenario)...")
diagnostic_results = kb.query(
    clinical_context={
        "symptoms": ["fever", "tachycardia", "hypotension"],
        "medications": ["metformin", "lisinopril"],
        "symptom_profile": ["fever_with_rigors", "urinary_symptoms"]
    },
    query_type="diagnostic"
)

print()
print("  Knowledge Base Results:")
print("  " + "-" * 60)
for i, result in enumerate(diagnostic_results, 1):
    if isinstance(result, dict):
        # Guideline results
        if "guideline" in result:
            print(f"  [{i}] Guideline: {result.get('guideline', 'N/A')}")
            print(f"      Recommendation: {result.get('recommendation', 'N/A')}")
            print(f"      Evidence Grade: {result.get('evidence_grade', 'N/A')}")
        # Drug interaction results
        elif "drug_pair" in result:
            pair = result["drug_pair"]
            print(f"  [{i}] Drug Interaction: {pair[0]} + {pair[1]}")
            print(f"      Severity: {result.get('severity', 'N/A')}")
            print(f"      Recommendation: {result.get('recommendation', 'N/A')}")
        # Provenance for all
        prov = result.get("provenance", {})
        print(f"      Provenance: source={prov.get('source', '?')}, "
              f"version={prov.get('version', '?')}, "
              f"confidence={prov.get('confidence', 0):.2f}")
        print()

# ─── Demo: Conflict resolution (§13.1, p. 367) ───
log_info("Demonstrating conflict resolution mechanism...")
print()
print("  Conflict Scenario: Two guidelines provide different antibiotic recommendations")
print("  " + "-" * 60)
print("  ┌─ Guideline A (IDSA 2024): Broad-spectrum empiric therapy within 1 hour")
print("  └─ Guideline B (WHO 2023):  Narrow-spectrum targeted therapy after cultures")
print()
print("  Resolution: CONFLICT FLAGGED — Both recommendations presented to clinician")
print("  → The agent amplifies clinical judgment rather than replacing it (§13.1, p. 367)")
print()
log_success("ClinicalKnowledgeBase demonstrated with provenance and conflict resolution.")


[INFO] Initializing ClinicalKnowledgeBase...
[INFO] [SIMULATION] DrugInteractionDB initialized | sources: ['drugbank', 'rxnorm', 'fda_labels']
[INFO] [SIMULATION] ClinicalGuidelineEngine initialized | sources: ['nice', 'who', 'aha', 'idsa']
[INFO] [SIMULATION] DiseaseOntology initialized | base: snomed_ct
[INFO] [SIMULATION] MedicalLiteratureIndex initialized | model: biomedical-bert

[INFO] Querying for diagnostic guidance (sepsis scenario)...
[INFO] [SIMULATION] ClinicalGuidelineEngine.search() | §13.1
[INFO] [SIMULATION] DrugInteractionDB.check_interactions() | §13.1
[SUCCESS] Step complete. query returned valid output.

  Knowledge Base Results:
  ------------------------------------------------------------
  [1] Guideline: Surviving Sepsis Campaign 2024
      Recommendation: Broad-spectrum antibiotics within 1 hour
      Evidence Grade: 1A
      Provenance: source=idsa, version=2024.1, confidence=0.96

  [2] Guideline: AHA 2024.2
      Recommendation: Monitor INR every 48h
      E

### 2c: FHIR Normalization & Patient Data Pipeline
*Ref: §13.2, pp. 367–369 — Multi-modal patient data processing with temporal alignment*

The pipeline handles heterogeneous clinical data through two stages:
1. **FHIRNormalizationLayer** — Converts HL7v2, CSV, Epic/Cerner formats into canonical FHIR R4 resources
2. **PatientDataPipeline** — Orchestrates biometric analysis, symptom interpretation, and history retrieval, then aligns all streams onto a unified temporal timeline

In [14]:
# Cell 2c — FHIRNormalizationLayer + PatientDataPipeline
# Ref: §13.2, pp. 367–369 — data ingestion with temporal alignment
# Author: Imran Ahmad

class FHIRNormalizationLayer:
    """
    Transforms heterogeneous clinical data formats into
    canonical FHIR R4 resources for downstream processing.

    Ref: §13.2, pp. 367–368
    Author: Imran Ahmad
    """

    def __init__(self):
        self.adapters = {
            "hl7v2": HL7v2ToFHIRAdapter(),
            "fhir_r4": PassthroughAdapter(),
            "csv_lab": CSVLabResultAdapter(),
            "epic_api": EpicFHIRAdapter(),
            "cerner_api": CernerFHIRAdapter()
        }
        self.validator = FHIRResourceValidator(
            profile="us-core-6.0"
        )

    @graceful_fallback(fallback_value=lambda: [MOCK_FHIR_BUNDLE], section_ref="13.2")
    def normalize(self, raw_data: dict, source_format: str = "fhir_r4") -> list:
        """
        Select adapter, transform data, validate against US Core profile.

        Ref: §13.2, pp. 367–368
        Author: Imran Ahmad
        """
        adapter = self.adapters.get(source_format)
        if not adapter:
            raise UnsupportedFormatError(
                f"No adapter for {source_format}"
            )

        # Extract individual resources from bundle
        if isinstance(raw_data, dict) and "entry" in raw_data:
            fhir_resources = [entry["resource"] for entry in raw_data["entry"]]
        else:
            fhir_resources = adapter.transform(raw_data)

        validated = []
        for resource in fhir_resources:
            result = self.validator.validate(resource)
            if result.is_valid:
                validated.append(resource)
            else:
                log_error(f"FHIR validation failed for {resource.get('resourceType', 'unknown')}: "
                         f"{result.errors}")

        return validated


class PatientDataPipeline:
    """
    Multi-modal patient data processing with temporal alignment
    and anomaly detection.

    Ref: §13.2, pp. 368–369
    Author: Imran Ahmad
    """

    def __init__(self):
        self.normalizer = FHIRNormalizationLayer()
        self.biometric_agent = BiometricAnalyzer(
            processors={
                "heart_rate": HeartRateProcessor(),
                "blood_pressure": BloodPressureProcessor(),
                "blood_glucose": BloodGlucoseProcessor(),
                "oxygen_saturation": SpO2Processor(),
                "activity": ActivityProcessor()
            }
        )
        self.symptom_agent = SymptomInterpreter(
            nlp_model="clinical-bert",
            symptom_ontology="medra"
        )
        self.history_agent = PatientHistoryAgent(
            memory_type="episodic",
            retention_policy="hipaa_compliant"
        )

    @graceful_fallback(fallback_value=lambda: MOCK_CLINICAL_CONTEXT, section_ref="13.2")
    def process(self, patient_id: str, current_data: dict,
                source_format: str = "fhir_r4") -> dict:
        """
        Orchestrate sub-agents and merge into temporally aligned context.

        Ref: §13.2, pp. 368–369
        Author: Imran Ahmad
        """
        normalized = self.normalizer.normalize(current_data, source_format)

        # Extract vitals from normalized resources
        vitals_data = {}
        for resource in normalized:
            if resource.get("resourceType") == "Observation":
                code_display = resource.get("code", {}).get("coding", [{}])[0].get("display", "")
                value = resource.get("valueQuantity", {}).get("value")
                if "temperature" in code_display.lower():
                    vitals_data["temperature_c"] = value
                elif "heart rate" in code_display.lower():
                    vitals_data["heart_rate_bpm"] = value
        # Supplement with direct vitals if available
        vitals_data.update({k: v for k, v in MOCK_PATIENT_VITALS.items()
                          if k != "_source" and k != "patient_id"})

        vitals = self.biometric_agent.analyze(vitals_data)
        symptoms = self.symptom_agent.interpret(
            ["fever_with_rigors", "tachycardia", "hypotension"],
            patient_context={"age": 67, "gender": "male"}
        )
        history = self.history_agent.retrieve(
            patient_id,
            relevance_window="36_months",
            priority_conditions=symptoms.get("flagged_conditions", [])
        )

        # Temporal alignment (§13.2, p. 369)
        timeline = self._align_temporal_data(vitals, symptoms, history)

        return {
            "patient_id": patient_id,
            "vitals": vitals,
            "symptoms": symptoms,
            "history": history,
            "timeline": timeline,
            "data_quality": self._assess_data_completeness(vitals, symptoms, history)
        }

    def _align_temporal_data(self, vitals, symptoms, history) -> list:
        """
        Correlate streams onto unified timeline.
        A fever preceding a cough by three days carries different
        diagnostic implications than one developing simultaneously.

        Ref: §13.2, p. 369
        """
        timeline = []
        timeline.append({
            "time": "2026-03-30T06:00:00Z",
            "event": "Fever onset (38.2°C)",
            "source": "patient_reported"
        })
        timeline.append({
            "time": "2026-03-30T08:00:00Z",
            "event": "Rigors reported, tachycardia noted",
            "source": "telehealth_consultation"
        })
        timeline.append({
            "time": "2026-03-30T10:30:00Z",
            "event": f"ED presentation — T:{vitals.get('temperature_c', '?')}°C, "
                     f"HR:{vitals.get('heart_rate_bpm', '?')} bpm, "
                     f"BP:{vitals.get('blood_pressure', '?')} mmHg",
            "source": "biometric_agent"
        })
        return timeline

    def _assess_data_completeness(self, vitals, symptoms, history) -> str:
        """
        Quantify uncertainty from missing data.
        Ref: §13.2, p. 370 — 'Clinical data is rarely complete'
        """
        completeness_score = 0
        if vitals and vitals.get("anomalies_detected"):
            completeness_score += 0.4
        if symptoms and symptoms.get("primary_symptoms"):
            completeness_score += 0.3
        if history and history.get("conditions"):
            completeness_score += 0.3
        if completeness_score >= 0.9:
            return "complete"
        elif completeness_score >= 0.6:
            return "partial — some data sources unavailable"
        else:
            return "incomplete — consider requesting additional records"


# ─── Demo: Normalize FHIR bundle ───
log_info("Initializing FHIRNormalizationLayer...")
normalizer = FHIRNormalizationLayer()

print()
log_info(f"Normalizing FHIR bundle ({len(MOCK_FHIR_BUNDLE['entry'])} resources)...")
validated = normalizer.normalize(MOCK_FHIR_BUNDLE, source_format="fhir_r4")

print()
print("  Validated FHIR Resources:")
print("  " + "-" * 50)
for r in validated:
    rtype = r.get("resourceType", "unknown")
    rid = r.get("id", "unknown")
    if rtype == "Patient":
        name = r.get("name", [{}])[0]
        print(f"  ✓ {rtype}: {name.get('given', [''])[0]} {name.get('family', '')} (id: {rid})")
    elif rtype == "Observation":
        display = r.get("code", {}).get("coding", [{}])[0].get("display", "")
        val = r.get("valueQuantity", {})
        print(f"  ✓ {rtype}: {display} = {val.get('value', '?')} {val.get('unit', '')}")
    elif rtype == "Condition":
        display = r.get("code", {}).get("coding", [{}])[0].get("display", "")
        print(f"  ✓ {rtype}: {display} (onset: {r.get('onsetDateTime', '?')})")

# ─── Demo: Full patient data pipeline ───
print()
log_info("Running PatientDataPipeline.process()...")
pipeline = PatientDataPipeline()
clinical_context = pipeline.process(
    patient_id="SIM-PT-00421",
    current_data=MOCK_FHIR_BUNDLE,
    source_format="fhir_r4"
)

print()
print("  Clinical Context Output:")
print("  " + "-" * 60)
print(f"  Patient ID:    {clinical_context['patient_id']}")
print(f"  Data Quality:  {clinical_context['data_quality']}")
print(f"  Anomalies:     {clinical_context['vitals'].get('anomalies_detected', [])}")
print(f"  Flagged:       {clinical_context['symptoms'].get('flagged_conditions', [])}")
print(f"  Hx Conditions: {clinical_context['history'].get('conditions', [])}")
print(f"  Hx Medications:{clinical_context['history'].get('medications', [])}")
print()
print("  Temporal Timeline:")
for event in clinical_context["timeline"]:
    print(f"    {event['time'][11:19]} | {event['source']:<25s} | {event['event']}")

print()
log_success("PatientDataPipeline demonstrated with FHIR normalization and temporal alignment.")


[INFO] Initializing FHIRNormalizationLayer...
[INFO] [SIMULATION] FHIRResourceValidator initialized | profile: us-core-6.0

[INFO] Normalizing FHIR bundle (4 resources)...
[INFO] [SIMULATION] Validating FHIR resource: Patient
[INFO] [SIMULATION] Validating FHIR resource: Observation
[INFO] [SIMULATION] Validating FHIR resource: Observation
[INFO] [SIMULATION] Validating FHIR resource: Condition
[SUCCESS] Step complete. normalize returned valid output.

  Validated FHIR Resources:
  --------------------------------------------------
  ✓ Patient: Test Simulation (id: SIM-PT-00421)
  ✓ Observation: Body temperature = 38.9 Cel
  ✓ Observation: Heart rate = 118 /min
  ✓ Condition: Type 2 diabetes mellitus (onset: 2018-04-01)

[INFO] Running PatientDataPipeline.process()...
[INFO] [SIMULATION] FHIRResourceValidator initialized | profile: us-core-6.0
[INFO] [SIMULATION] BiometricAnalyzer initialized | §13.2
[INFO] [SIMULATION] SymptomInterpreter initialized | model: clinical-bert
[INFO] [SIMU

### 2d: Diagnostic Coordinator — Full Pipeline
*Ref: §13.3, pp. 370–374 — End-to-end clinical decision support*

The `DiagnosticCoordinator` orchestrates the complete diagnostic pipeline:
1. Evidence gathering (biometrics, symptoms, episodic memory)
2. Differential diagnosis generation with ranked candidates
3. Platt-calibrated confidence scoring
4. Safety escalation for critical conditions (threshold: 0.15)
5. Audience-adapted explanation (clinician SHAP report vs. patient summary)
6. Immutable audit trail recording

The escalation threshold of **0.15** emerged from cost-asymmetry analysis: the cost of a missed myocardial infarction or sepsis case outweighs a false alarm by roughly an order of magnitude (§13.3, p. 373).

In [15]:
# Cell 2d — DiagnosticCoordinator: Full Clinical Decision Pipeline
# Ref: §13.3, pp. 370–374 — end-to-end pipeline with safety escalation
# Author: Imran Ahmad

class DiagnosticCoordinator:
    """
    Clinical decision support with Bayesian evidence integration
    and confidence-calibrated differential diagnosis.

    Ref: §13.3, pp. 370–373
    Author: Imran Ahmad
    """

    def __init__(self):
        self.biometric_agent = BiometricAnalyzer()
        self.symptom_agent = SymptomInterpreter()
        self.coordinator = DifferentialGenerator()
        self.explainer = ClinicalExplainer()
        self.confidence_engine = ConfidenceAwareAgent(
            n_hypotheses=5,
            calibration_method="platt_scaling"
        )
        self.memory = ClinicalMemorySystem()
        self.safety_monitor = SafetyMonitor(
            escalation_threshold=0.15,
            critical_conditions=[
                "myocardial_infarction",
                "pulmonary_embolism",
                "sepsis",
                "stroke"
            ]
        )
        self.audit_trail = AuditTrailGenerator()
        self._kb_version = "2026.Q1"
        self._model_version = "clinical-v3.2"

    @graceful_fallback(fallback_value=lambda: MOCK_DIAGNOSTIC_REPORT, section_ref="13.3")
    def diagnose(self, patient_data: dict, reported_symptoms: list) -> dict:
        """
        Generate an explained, confidence-rated diagnosis.

        Ref: §13.3, pp. 370–373 — complete pipeline sequence
        Author: Imran Ahmad
        """
        # Phase 1: Gather evidence
        vitals = self.biometric_agent.analyze(patient_data)
        symptoms = self.symptom_agent.interpret(
            reported_symptoms,
            patient_context=patient_data
        )
        history = self.memory.retrieve_episodic(
            patient_data.get("patient_id", "unknown")
        )

        # Phase 2: Generate and score differentials
        differentials = self.coordinator.generate_differentials(
            vitals, symptoms, history
        )
        scored = self.confidence_engine.score_differentials(
            differentials,
            evidence={
                "vitals": vitals,
                "symptoms": symptoms,
                "history": history
            }
        )

        # Phase 3: Safety evaluation (§13.3, pp. 373–374)
        safety_result = self.safety_monitor.evaluate(scored)
        if safety_result.requires_escalation:
            self.safety_monitor.trigger_alert(
                patient_data.get("patient_id", "unknown"),
                safety_result
            )

        # Phase 4: Generate audience-adapted explanations (§13.3, p. 374)
        clinician_explanation = self.explainer.generate(
            scored,
            audience="clinician",
            evidence_sources=[vitals, symptoms, history]
        )
        patient_explanation = self.explainer.generate(
            scored,
            audience="patient",
            evidence_sources=[vitals, symptoms, history]
        )

        # Phase 5: Store episodic memory
        self.memory.store_episodic(patient_data.get("patient_id", "unknown"), {
            "diagnoses": scored,
            "evidence": [vitals, symptoms],
            "explanation": clinician_explanation
        })

        # Phase 6: Record immutable audit trail (§13.3, p. 373)
        audit_id = self.audit_trail.record(InferenceEvent(
            patient_id=patient_data.get("patient_id", "unknown"),
            inputs=["vitals", "symptoms", "history"],
            kb_version=self._kb_version,
            model_version=self._model_version,
            reasoning_steps=self.coordinator.get_trace(),
            output=[{d["diagnosis"]: d["calibrated_confidence"]} for d in scored],
            confidence_breakdown=self.confidence_engine.get_calibration_report(),
            safety_alerts=safety_result.alerts
        ))

        return {
            "differentials": scored,
            "clinician_explanation": clinician_explanation,
            "patient_explanation": patient_explanation,
            "confidence_summary": self.confidence_engine.communicate_uncertainty(scored),
            "safety_result": {
                "escalation_required": safety_result.requires_escalation,
                "alerts": safety_result.alerts,
                "threshold": safety_result.threshold_used
            },
            "audit_trail_id": audit_id,
            "reasoning_trace": self.coordinator.get_trace()
        }

    def get_knowledge_base_version(self):
        return self._kb_version

    def get_model_version(self):
        return self._model_version


# ═══════════════════════════════════════════════════════════════
# DEMO: Full Diagnostic Pipeline
# ═══════════════════════════════════════════════════════════════

log_info("Initializing DiagnosticCoordinator...")
coordinator = DiagnosticCoordinator()

print()
log_info("Running full diagnostic pipeline on sepsis scenario...")
print()

report = coordinator.diagnose(
    patient_data=MOCK_PATIENT_VITALS,
    reported_symptoms=["fever_with_rigors", "tachycardia", "hypotension",
                       "urinary_burning", "confusion"]
)

# ─── Display: Differential Ranking ───
print("=" * 70)
print("  DIAGNOSTIC REPORT")
print("=" * 70)
print()
print("  Ranked Differential Diagnosis:")
print("  " + "-" * 60)
for i, d in enumerate(report["differentials"], 1):
    conf = d["calibrated_confidence"]
    bar = "█" * int(conf * 40) + "░" * (40 - int(conf * 40))
    print(f"  {i}. {d['diagnosis']:<22s} {bar} {conf:.3f}")
print()

# ─── Display: Confidence Summary ───
cs = report["confidence_summary"]
print(f"  Confidence Summary: {cs['top_diagnosis']} at {cs['confidence']:.2f} "
      f"({cs['interpretation']}) — method: {cs['calibration_method']}")
print()

# ─── Display: Safety Escalation ───
sr = report["safety_result"]
if sr["escalation_required"]:
    print(f"  ⚠ SAFETY ESCALATION TRIGGERED (threshold: {sr['threshold']})")
    for alert in sr["alerts"]:
        print(f"    → {alert['condition']} (confidence: {alert['confidence']:.2f}) — "
              f"{alert['action']}")
else:
    print("  ✓ No safety escalation required.")
print()

# ─── Display: Reasoning Trace ───
print("  Reasoning Trace:")
for step in report["reasoning_trace"]:
    print(f"    • {step}")
print()

# ─── Display: Clinician Explanation (§13.3, p. 374) ───
print("  ┌─ CLINICIAN EXPLANATION (SHAP-attributed)")
print("  │")
for line in report["clinician_explanation"].split(". "):
    if line.strip():
        print(f"  │  {line.strip()}.")
print("  └─")
print()

# ─── Display: Patient Explanation (§13.3, p. 374) ───
print("  ┌─ PATIENT EXPLANATION")
print("  │")
for line in report["patient_explanation"].split(". "):
    if line.strip():
        print(f"  │  {line.strip()}.")
print("  └─")
print()

# ─── Display: Audit Trail ───
print(f"  Audit Trail ID: {report['audit_trail_id']}")
print(f"  KB Version: {coordinator.get_knowledge_base_version()}")
print(f"  Model Version: {coordinator.get_model_version()}")
print()

log_success("DiagnosticCoordinator full pipeline demonstrated successfully.")
print("=" * 70)


[INFO] Initializing DiagnosticCoordinator...
[INFO] [SIMULATION] BiometricAnalyzer initialized | §13.2
[INFO] [SIMULATION] SymptomInterpreter initialized | model: clinical-bert
[INFO] [SIMULATION] DifferentialGenerator initialized | §13.3
[INFO] [SIMULATION] ClinicalExplainer initialized | §13.3
[INFO] [SIMULATION] ConfidenceAwareAgent initialized | method: platt_scaling
[INFO] [SIMULATION] ClinicalMemorySystem initialized | §13.3
[INFO] [SIMULATION] SafetyMonitor initialized | threshold: 0.15 | critical conditions: ['myocardial_infarction', 'pulmonary_embolism', 'sepsis', 'stroke']
[INFO] [SIMULATION] AuditTrailGenerator initialized | §13.3

[INFO] Running full diagnostic pipeline on sepsis scenario...

[INFO] [SIMULATION] BiometricAnalyzer.analyze() | §13.2
[INFO] [SIMULATION] SymptomInterpreter.interpret() | §13.2
[INFO] [SIMULATION] ClinicalMemorySystem.retrieve_episodic(SIM-PT-00421) | §13.3
[INFO] [SIMULATION] DifferentialGenerator.generate_differentials() | §13.3
[INFO] [SIMULAT

### 2e: Case Study — Diagnostic Assistance System
*Ref: §13.4, pp. 374–375 — Regional health network deployment results*

A regional health network (200,000 patients, 20 provider sites) deployed a multi-agent diagnostic assistant for catching chronic condition flare-ups before ED presentation. Key architectural features: edge computing with differential privacy (ε = 1.0), audience-adapted explanations, and explicit safety escalation.

In [16]:
# Cell 2e — Case Study: Deployment Metrics Visualization
# Ref: §13.4, pp. 374–375 — Diagnostic Assistance System results
# Author: Imran Ahmad

log_info("Displaying deployment metrics from §13.4 case study...")
print()

m = MOCK_DEPLOYMENT_METRICS

print("=" * 70)
print("  CASE STUDY: DIAGNOSTIC ASSISTANCE SYSTEM")
print("  Regional Health Network Deployment Results")
print("=" * 70)
print()

# ─── Deployment Scale ───
print("  Deployment Scale:")
print(f"    Patient population:  {m['patient_population']:>10,}")
print(f"    Provider sites:      {m['provider_sites']:>10}")
print()

# ─── Clinical Outcomes ───
print("  Clinical Outcomes:")
print("  " + "-" * 55)

metrics = [
    ("Early Detection Improvement", f"+{m['early_detection_improvement_pct']}%",
     "Patients caught during routine visits vs. ED presentation"),
    ("False Alarm Rate", f"{m['false_alarm_rate_pct']}%",
     "Critical for adoption — avoids alert fatigue"),
    ("Clinician Response Time", f"+{m['clinician_response_improvement_pct']}% faster",
     "Structured reports reduce info-gathering time"),
    ("Physician Satisfaction", f"{m['physician_satisfaction_pct']}%",
     "Transparent reasoning traces increase trust"),
]

for name, value, note in metrics:
    print(f"    {name:<30s}  {value:>12s}")
    print(f"      → {note}")
print()

# ─── Privacy Architecture ───
print("  Privacy Architecture (Edge Computing + Differential Privacy):")
print("  " + "-" * 55)
print(f"    Differential privacy epsilon:  {m['differential_privacy_epsilon']}")
print(f"    Raw data rate:                 {m['raw_data_rate_kb_per_sec']} KB/sec per patient")
print(f"    Derived feature size:          {m['derived_feature_size_bytes']} bytes every {m['transmission_interval_min']} min")
print()

# Data transmission comparison (§13.4, p. 375)
patients = 10_000
raw_daily_tb = (m["raw_data_rate_kb_per_sec"] * 1000 * 86400 * patients) / (1e12)
derived_daily_mb = (m["derived_feature_size_bytes"] * (1440 / m["transmission_interval_min"])
                    * patients) / (1e6)
print(f"    For {patients:,} simultaneous patients:")
print(f"      Raw data:     ~{raw_daily_tb:.1f} TB/day")
print(f"      Derived only: ~{derived_daily_mb:.0f} MB/day")
print(f"      Reduction:    ~{raw_daily_tb * 1e6 / derived_daily_mb:.0f}x")
print()

# ─── Key Lessons ───
print("  Key Architectural Lessons (§13.4, p. 375):")
print("    1. Edge computing + differential privacy enables regulatory compliance")
print("    2. Audience-adapted explanations serve clinicians AND patients")
print("    3. Safety escalation triggers immediate alerts regardless of confidence")
print()
log_success("Case study metrics displayed. See §13.4 (pp. 374–375) for full analysis.")
print("=" * 70)


[INFO] Displaying deployment metrics from §13.4 case study...

  CASE STUDY: DIAGNOSTIC ASSISTANCE SYSTEM
  Regional Health Network Deployment Results

  Deployment Scale:
    Patient population:     200,000
    Provider sites:              20

  Clinical Outcomes:
  -------------------------------------------------------
    Early Detection Improvement             +30%
      → Patients caught during routine visits vs. ED presentation
    False Alarm Rate                          3%
      → Critical for adoption — avoids alert fatigue
    Clinician Response Time          +40% faster
      → Structured reports reduce info-gathering time
    Physician Satisfaction                   92%
      → Transparent reasoning traces increase trust

  Privacy Architecture (Edge Computing + Differential Privacy):
  -------------------------------------------------------
    Differential privacy epsilon:  1.0
    Raw data rate:                 4 KB/sec per patient
    Derived feature size:          20

## Summary

- **Four-layer architecture** (§13.1) — data ingestion, clinical knowledge, reasoning and decision, and explanation and delivery communicate through well-defined interfaces, so knowledge bases can be updated without disturbing the reasoning pipeline.
- **Bayesian belief updating** (§13.1) maintains a posterior distribution over candidate diagnoses, renormalized as each new observation arrives — the POMDP belief state made concrete.
- **Safety escalation** (§13.3) triggers immediate clinician alerts whenever a critical condition exceeds the 0.15 confidence threshold — a deliberate cost-asymmetry tradeoff favoring false alarms over missed diagnoses.
- **Audience-adapted explanations** (§13.3) surface SHAP-attributed reasoning for clinicians and plain-language summaries for patients, with every inference recorded in an immutable audit trail.
- **Deployment case study** (§13.4) — a regional health network (200,000 patients, 20 provider sites) reported 30% better early detection, a 3% false-alarm rate, and 92% physician satisfaction.

**Next:** See Chapter 13 (§13.1–13.4, pp. 362–375) of *30 Agents Every AI Engineer Must Build* for the full architectural discussion, and §13.5–13.8 for the companion Scientific Discovery Agent.
